# Véletlenszerű játszma Stockfish-elemzése + LLM narráció API hívással

**Pipeline:**
1. Véletlenszerű játszma sorsolása a `mychessdotcomgames.parquet` fájlból
2. Stockfish elemzés (centipawn annotációk lépésenként)
3. Annotált PGN exportálása → `output/llm-analysis/elemzett.pgn`
4. LLM API: didaktikus narráció generálása (provider: `config.LLM_PROVIDER`)
5. Narráció mentése → `output/llm-analysis/{llm_provider}_{game_id}.txt`

In [1]:
import os
import sys
import random
import shutil

# ROOT_DIR: a könyvtár ahol a config.py van
_cwd = os.getcwd()
ROOT_DIR = _cwd if os.path.exists(os.path.join(_cwd, "config.py"))            else os.path.abspath(os.path.join(_cwd, ".."))
sys.path.insert(0, ROOT_DIR)

import polars as pl
import chess
import chess.pgn
import chess.engine

import config
from src.llm_client import generate_text

print(f"ROOT_DIR    : {ROOT_DIR}")
print(f"LLM provider: {config.LLM_PROVIDER}  ({config.LLM_MODEL})")
print(f"API kulcs   : {chr(39)+chr(79)+chr(75)+chr(39) if config.LLM_API_KEY else chr(39)+'HIÁNYZIK – töltsd ki a secrets.py-t!'+chr(39)}")

ROOT_DIR    : D:\Workspace\chess-pgn-analysis
LLM provider: openai  (gpt-4o-mini)
API kulcs   : 'OK'


## 1. Véletlen játszma kiválasztása

In [2]:
PARQUET_PATH = os.path.join(ROOT_DIR, "output", "parquet", "mychessdotcomgames.parquet")

df = pl.read_parquet(PARQUET_PATH)
print(f"Összes játszma a parquet-ban: {len(df):,}")

idx = random.randint(0, len(df) - 1)
row = df.row(idx, named=True)

print(f"\nKiválasztott játszma (sor-index={idx}, game_id={row['game_id']}):")
print(f"  Fehér  : {row['white']} ({row['white_elo']})")
print(f"  Fekete : {row['black']} ({row['black_elo']})")
print(f"  Megnyitó: {row['eco']} · {row['opening']}")
print(f"  Eredmény: {row['result']}  |  Lépések: {row['num_moves']}")

Összes játszma a parquet-ban: 1,377

Kiválasztott játszma (sor-index=666, game_id=667):
  Fehér  : Wujajin (1029)
  Fekete : NurgeldiAssan (1040)
  Megnyitó: D00 · 
  Eredmény: 0-1  |  Lépések: 49


## 2. Stockfish elemzés

A kisebb depth gyorsabb futást eredményez, de a magasabb alaposabb (pontosabb) elemzést!

In [3]:
def find_stockfish() -> str:
    if getattr(config, 'STOCKFISH_PATH', None) and os.path.isfile(config.STOCKFISH_PATH):
        return config.STOCKFISH_PATH
    bundled = os.path.join(ROOT_DIR, "stockfish", "stockfish-windows-x86-64-avx2.exe")
    if os.path.isfile(bundled):
        return bundled
    found = shutil.which("stockfish")
    if found:
        return found
    raise FileNotFoundError("Stockfish nem található! Ellenőrizd a stockfish/ mappát.")

SF_PATH = find_stockfish()
print(f"Stockfish: {SF_PATH}")

Stockfish: D:\Workspace\chess-pgn-analysis\stockfish\stockfish-windows-x86-64-avx2.exe


In [4]:
import asyncio
from tqdm.notebook import tqdm

# Windows + Python 3.9 alatt a Jupyter SelectorEventLoop-ot használ,
# ami nem tud subprocesst indítani – a chess.engine ezt igényli.
if hasattr(asyncio, 'WindowsProactorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Depth csökkentése gyorsabb futáshoz (config: 18, teszt: 12)
DEPTH       = 12
MOVES_LIMIT = config.STOCKFISH_MOVES_LIMIT

moves_uci_list = row['moves_uci'].strip().split()
to_analyze     = min(len(moves_uci_list), MOVES_LIMIT)
print(f"Elemzés: mélység={DEPTH}, lépések={to_analyze}/{len(moves_uci_list)}\n")

board       = chess.Board()
evaluations = []

with chess.engine.SimpleEngine.popen_uci(SF_PATH) as engine:
    limit = chess.engine.Limit(depth=DEPTH)

    for uci in tqdm(moves_uci_list[:MOVES_LIMIT], desc="Stockfish", unit="lépés"):
        move = chess.Move.from_uci(uci)
        if move not in board.legal_moves:
            print(f"  Illegális lépés: {uci} – leállítás")
            break

        san = board.san(move)
        board.push(move)

        info  = engine.analyse(board, limit)
        score = info["score"].white()

        evaluations.append({
            "uci":  uci,
            "san":  san,
            "cp":   score.score(mate_score=10000),
            "mate": score.mate(),
        })

print(f"\nKész: {len(evaluations)} lépés elemezve.")

Elemzés: mélység=12, lépések=40/49



Stockfish:   0%|          | 0/40 [00:00<?, ?lépés/s]


Kész: 40 lépés elemezve.


## 3. Annotált PGN exportálása

In [ ]:
game_id   = row["game_id"]
os.makedirs(config.LLM_ANALYSIS_DIR, exist_ok=True)
PGN_PATH  = config.ELEMZETT_PGN

# Duplikátum-ellenőrzés
pgn_exists     = os.path.exists(PGN_PATH) and os.path.getsize(PGN_PATH) > 0
already_in_pgn = False
if pgn_exists:
    with open(PGN_PATH, encoding="utf-8") as f:
        if f'[ParquetGameId "{game_id}"]' in f.read():
            already_in_pgn = True

if already_in_pgn:
    print(f"⚠️  Játszma #{game_id} már szerepel az elemzett.pgn-ben – kihagyva.")
else:
    game_pgn = chess.pgn.Game()
    game_pgn.headers.update({
        "Event":         row.get("event",    "?"),
        "Site":          row.get("site",     "?"),
        "Date":          row.get("date",     "?"),
        "White":         row.get("white",    "?"),
        "Black":         row.get("black",    "?"),
        "Result":        row.get("result",   "*"),
        "WhiteElo":      str(row.get("white_elo", "?")),
        "BlackElo":      str(row.get("black_elo", "?")),
        "ECO":           row.get("eco",      "?"),
        "Opening":       row.get("opening",  "?"),
        "ParquetGameId": str(game_id),
    })

    node = game_pgn
    for ev in evaluations:
        node = node.add_variation(chess.Move.from_uci(ev["uci"]))
        if ev["mate"] is not None:
            node.comment = f"[%eval #{ev['mate']}]"
        elif ev["cp"] is not None:
            node.comment = f"[%eval {ev['cp'] / 100:.2f}]"

    # StringExporter garantál egységes kimenetet; .strip()+"\n" pontosan egy \n-re zárja a játszmát.
    # Ha már van tartalom, egy "\n" elé kerül → mindig pontosan egy üres sor lesz a játszmák között.
    pgn_str = game_pgn.accept(chess.pgn.StringExporter(headers=True, variations=True, comments=True))
    with open(PGN_PATH, "a", encoding="utf-8") as f:
        if pgn_exists:
            f.write("\n")
        f.write(pgn_str.strip() + "\n")

    with open(PGN_PATH, encoding="utf-8") as f:
        content = f.read()
    game_count = content.count("[Event ")
    print(f"PGN hozzáfűzve: {PGN_PATH}")
    print(f"Összes elemzett játszma: {game_count}")
    print("\nUtolsó hozzáfűzött játszma (első 600 karakter):")
    last_start = content.rfind("[Event ")
    print(content[last_start:last_start + 600])

## 4. LLM narráció generálása

In [6]:
NARRATION_PROMPT = (
    "Készíts 2 bekezdésben didaktív szöveges narrációt a csatolt .pgn sakkjátszma alapján, "
    "ügyelj rá, hogy a figurák nevét jól írd ki (mivel hangosan fel lesz olvasva), "
    "amikor egy lépésre hivatkozol, pl. Rxh7-et így írd: bástya üti h7-et, stb.! "
    "A narrációd célja az oktatás és hogy döntő kulcspillanatokat kiemelj!"
)

with open(PGN_PATH, encoding="utf-8") as f:
    pgn_text = f.read()

full_prompt = NARRATION_PROMPT + chr(10) * 2 + pgn_text
narration = generate_text(full_prompt)

print("--- LLM narráció ---")
print(narration)

--- LLM narráció ---
Az első játszmában a fehér játékos, Wujajin a d4 nyitásával indít, amelyre a fekete játékos, Louisdortbien d5-tel válaszol. Az első néhány lépés során a fehér játékos a futót Bf4-re mozgatja, hogy támogassa a középjátéki terveit, míg a fekete részéről a futó Bf5-re lép, a vezérbástyája fejlesztését célozva. Az igazi fordulópontot a hetedik lépés hozza, amikor a fehér vezér Qb5+ megadja az első sakkot, ami arra kényszeríti a feketét, hogy a c6-os gyalogot lépje, így a sakk elhárul. Ezután a vezér sikeresen elfogja a b4-es gyalogot, így egy anyagi előnyhöz jut.

A játszma folytatásában a fehér játékos aktívan nyomul előre, míg a fekete kissé védekezik. A kulcsfontosságú lépés a tizenötödik lépés, amikor a fehér bástya Rb1-re lép, készülve egy erős pozíció kiépítésére. A fekete válasza Qc5-tel a saját vezérének elhelyezését célozza, de az álmaik szertefoszlanak, amikor Wujajin így megnyeri a játszmát, hiszen a helyzet egyértelműen kedvezővé válik a fehér számára. Ez a

## 5. Narráció mentése

In [7]:
llm_name = config.LLM_PROVIDER
txt_name = f"{llm_name}_{game_id}.txt"
LLM_PATH = os.path.join(config.LLM_ANALYSIS_DIR, txt_name)

if os.path.exists(LLM_PATH):
    print(f"⚠️  Narráció már létezik: {LLM_PATH} – kihagyva.")
else:
    with open(LLM_PATH, "w", encoding="utf-8") as f:
        f.write(narration)
    print(f"Narráció elmentve: {LLM_PATH}")

Narráció elmentve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\openai_667.txt
